# 04 Win Rate — Logistic Regression

## Goal

For every `(character, card)` pair in `gold_card_choice_events`, restrict to occasions where the card was offered and fit

```
victory ~ was_picked + hp_ratio + floor + relic_count + ascension_level
```

`was_picked`'s coefficient is the card's effect on win probability *holding run state at the time of the offer constant* — a cleaner signal than the raw pick/no-pick lift computed in 03, which doesn't control for anything.

This fits every card directly rather than pre-selecting a curated "cards of interest" list by raw lift and then re-estimating on the same data. That selection step is a winner's-curse setup: any card's raw lift is true effect plus sampling noise, and sorting by it preferentially keeps cards that got a lucky noise draw — that noise doesn't average out on a second look at the same rows, so the regression would end up biased toward whatever the screen happened to reward. Fitting every card sidesteps the problem entirely instead of working around it with a data split. `03`'s raw-lift screen is still worth checking against the results here as a sanity check (a card with strong raw lift but a controlled odds ratio near 1 suggests the raw lift was confounded), just not as a required input.

**Deliberately excluded:** `floor_reached` and `floors_gained` are not covariates here, even though they're in the table. Both are facts about how the run *ended*, not facts known at the time of the pick — `floor_reached` is close to deterministic of `victory` (a run that reaches floor 57 essentially won), so including it would leak the outcome into the predictors rather than control for a legitimate confounder. `floor` (the pick's own floor, i.e. `choice_floor`) is fine to include — that's "how far into the run this decision happened," known at decision time.

In [4]:
import os
from pathlib import Path

# Jupyter's cwd is the notebook's directory; project root is one level up.
PROJECT_ROOT = Path.cwd().parent

os.environ["JAVA_HOME"] = str(PROJECT_ROOT / ".jdk17" / "jdk-17.0.20+8")
os.environ["HADOOP_HOME"] = r"C:\hadoop"
os.environ["PATH"] = str(PROJECT_ROOT / ".venv" / "Scripts") + os.pathsep + r"C:\hadoop\bin" + os.pathsep + os.environ["PATH"]
os.environ["PYSPARK_PYTHON"] = str(PROJECT_ROOT / ".venv" / "Scripts" / "python.exe")
os.environ["PYSPARK_DRIVER_PYTHON"] = str(PROJECT_ROOT / ".venv" / "Scripts" / "python.exe")

GOLD_CARD_CHOICE_EVENTS_PATH = str(PROJECT_ROOT / "raw_data" / "gold" / "card_choice_events")

In [5]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from delta import configure_spark_with_delta_pip
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

builder = (
    SparkSession.builder.master("local[*]")
    .appName("win-rate-logistic-regression")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.driver.memory", "16g")
    .config("spark.driver.maxResultSize", "6g")
    .config("spark.sql.shuffle.partitions", "100")
)
spark = configure_spark_with_delta_pip(builder).getOrCreate()

df = spark.read.format("delta").load(GOLD_CARD_CHOICE_EVENTS_PATH)
print(f"Loaded {GOLD_CARD_CHOICE_EVENTS_PATH}")
spark.conf.set("spark.sql.execution.arrow.pyspark.enabled", "true")
print("Arrow enabled:", spark.conf.get("spark.sql.execution.arrow.pyspark.enabled"))

Loaded e:\Projects\sts-card-choice-analysis\raw_data\gold\card_choice_events
Arrow enabled: true


In [6]:
# Collect per character rather than the whole table in one toPandas() — a single collect
# over every card blew past both spark.driver.maxResultSize and, once that was raised, the
# JVM driver heap itself (collect() stages all rows as boxed Java objects before handing
# them to Python, which is much heavier than the on-disk Parquet size). Card pools are
# already partitioned by character_chosen, so chunking on it is free and bounds peak driver
# memory to roughly one character's worth of rows at a time.
characters = sorted(
    row["character_chosen"]
    for row in df.select("character_chosen").distinct().collect()
    if row["character_chosen"] is not None
)
print("Characters:", characters)

regression_pd_chunks = []
for character in characters:
    chunk = (
        df.filter(F.col("character_chosen") == character)
        .select("character_chosen", "card_name", "was_picked", "victory", "floor", "current_hp", "max_hp", "relic_count", "ascension_level")
        .na.drop()
        .toPandas()
    )
    print(f"{character}: collected {len(chunk)} rows")
    regression_pd_chunks.append(chunk)

regression_pd = pd.concat(regression_pd_chunks, ignore_index=True)
del regression_pd_chunks
print("Collected rows:", len(regression_pd))
print("Distinct (character, card) pairs:", regression_pd[["character_chosen", "card_name"]].drop_duplicates().shape[0])

Characters: ['DEFECT', 'IRONCLAD', 'THE_SILENT', 'WATCHER']
DEFECT: collected 54242228 rows
IRONCLAD: collected 78113867 rows
THE_SILENT: collected 60027797 rows
WATCHER: collected 38082393 rows
Collected rows: 230466285
Distinct (character, card) pairs: 3139


In [7]:
regression_pd["was_picked"] = regression_pd["was_picked"].astype(int)
regression_pd["victory"] = regression_pd["victory"].astype(int)
regression_pd = regression_pd[regression_pd["max_hp"] > 0].copy()
regression_pd["hp_ratio"] = regression_pd["current_hp"] / regression_pd["max_hp"]

MIN_REGRESSION_ROWS = 200

# groupby().apply() needs every group to return a Series with the same keys — a group
# returning fewer keys than another (e.g. just n/error on failure) makes pandas fall back
# to a stacked long-format result instead of one row per group, so every branch below
# fills the full set of keys even when most are None.
EMPTY_RESULT = {
    "n": None,
    "error": None,
    "was_picked_coef": None,
    "was_picked_pvalue": None,
    "odds_ratio": None,
    "odds_ratio_ci_low": None,
    "odds_ratio_ci_high": None,
}

def fit_card_logit(group):
    result = dict(EMPTY_RESULT)
    if len(group) < MIN_REGRESSION_ROWS:
        result["n"] = len(group)
        result["error"] = "too few rows"
        return pd.Series(result)
    try:
        model = smf.logit(
            "victory ~ was_picked + hp_ratio + floor + relic_count + ascension_level",
            data=group,
        ).fit(disp=0)
    except Exception as exc:
        result["n"] = len(group)
        result["error"] = str(exc)
        return pd.Series(result)
    coef = model.params["was_picked"]
    ci_low, ci_high = model.conf_int().loc["was_picked"]
    result.update({
        "n": len(group),
        "error": None,
        "was_picked_coef": coef,
        "was_picked_pvalue": model.pvalues["was_picked"],
        "odds_ratio": np.exp(coef),
        "odds_ratio_ci_low": np.exp(ci_low),
        "odds_ratio_ci_high": np.exp(ci_high),
    })
    return pd.Series(result)

results_pd = (
    regression_pd.groupby(["character_chosen", "card_name"])
    .apply(fit_card_logit, include_groups=False)
    .reset_index()
)
print("Fitted:", (results_pd["error"].isna()).sum(), "of", len(results_pd))

e:\Projects\sts-card-choice-analysis\.venv\Lib\site-packages\statsmodels\discrete\discrete_model.py:2385: RuntimeWarning: overflow encountered in exp
  return 1/(1+np.exp(-X))
e:\Projects\sts-card-choice-analysis\.venv\Lib\site-packages\statsmodels\discrete\discrete_model.py:2443: RuntimeWarning: divide by zero encountered in log
  return np.sum(np.log(self.cdf(q * linpred)))
e:\Projects\sts-card-choice-analysis\.venv\Lib\site-packages\statsmodels\discrete\discrete_model.py:2385: RuntimeWarning: overflow encountered in exp
  return 1/(1+np.exp(-X))
e:\Projects\sts-card-choice-analysis\.venv\Lib\site-packages\statsmodels\discrete\discrete_model.py:2443: RuntimeWarning: divide by zero encountered in log
  return np.sum(np.log(self.cdf(q * linpred)))
e:\Projects\sts-card-choice-analysis\.venv\Lib\site-packages\statsmodels\discrete\discrete_model.py:2385: RuntimeWarning: overflow encountered in exp
  return 1/(1+np.exp(-X))
e:\Projects\sts-card-choice-analysis\.venv\Lib\site-packages\stats

Fitted: 2232 of 3139


### Results

`odds_ratio` > 1 means picking the card is associated with higher win odds after controlling for HP ratio, floor, relic count, and ascension at the time it was offered; < 1 means lower. Sorted within each character by odds ratio, restricted to statistically significant results (`was_picked_pvalue < 0.05`) — cards that didn't reach significance at this sample size are dropped from this view but still available in `results_pd`.

In [8]:
significant = results_pd[
    results_pd["error"].isna() & (results_pd["was_picked_pvalue"] < 0.05)
].sort_values(["character_chosen", "odds_ratio"], ascending=[True, False])

cols = ["character_chosen", "card_name", "n", "odds_ratio", "odds_ratio_ci_low", "odds_ratio_ci_high", "was_picked_pvalue"]
significant[cols]

,character_chosen,card_name,n,odds_ratio,odds_ratio_ci_low,odds_ratio_ci_high,was_picked_pvalue
142,DEFECT,Concentrate,313.0,3.876976,1.566486,9.595329,3.382009e-03
95,DEFECT,Burning Pact,378.0,3.460880,1.667772,7.181854,8.585789e-04
443,DEFECT,Night Terror,343.0,2.357926,1.272884,4.367888,6.390193e-03
44,DEFECT,Barricade,344.0,1.956129,1.108846,3.450831,2.051981e-02
406,DEFECT,Master of Strategy+1,1206.0,1.947548,1.506740,2.517317,3.562503e-07
...,...,...,...,...,...,...,...
2397,WATCHER,All Out Attack,341.0,0.263159,0.084298,0.821518,2.153613e-02
3024,WATCHER,Storm+1,201.0,0.240441,0.065652,0.880584,3.139951e-02
2558,WATCHER,Darkness,320.0,0.204468,0.044484,0.939825,4.137894e-02
2701,WATCHER,Glacier+1,217.0,0.180768,0.067708,0.482616,6.401173e-04


## Stop Spark

Run this when done exploring — otherwise the JVM stays alive holding memory until the kernel is restarted.

In [7]:
spark.stop()

In [9]:
len(results_pd)

3139

In [11]:
significant = results_pd[
    results_pd["error"].isna() & (results_pd["was_picked_pvalue"] < 0.05)
].sort_values(["character_chosen", "odds_ratio"], ascending=[True, False])
print(len(significant), "significant of", (results_pd["error"].isna()).sum(), "fitted")
cols = ["character_chosen", "card_name", "n", "odds_ratio", "odds_ratio_ci_low", "odds_ratio_ci_high", "was_picked_pvalue"]
significant[cols].to_string(index=False)

1049 significant of 2232 fitted


'character_chosen            card_name         n  odds_ratio  odds_ratio_ci_low  odds_ratio_ci_high  was_picked_pvalue\n          DEFECT          Concentrate     313.0    3.876976           1.566486            9.595329       3.382009e-03\n          DEFECT         Burning Pact     378.0    3.460880           1.667772            7.181854       8.585789e-04\n          DEFECT         Night Terror     343.0    2.357926           1.272884            4.367888       6.390193e-03\n          DEFECT            Barricade     344.0    1.956129           1.108846            3.450831       2.051981e-02\n          DEFECT Master of Strategy+1    1206.0    1.947548           1.506740            2.517317       3.562503e-07\n          DEFECT           Dual Wield     401.0    1.947040           1.021049            3.712814       4.305105e-02\n          DEFECT    Calculated Gamble     373.0    1.896975           1.030519            3.491945       3.973196e-02\n          DEFECT              Glacier  431375.0

In [12]:
summary = (
    significant.groupby("character_chosen")
    .agg(n_significant=("card_name", "count"),
         n_positive=("odds_ratio", lambda s: (s > 1).sum()),
         n_negative=("odds_ratio", lambda s: (s < 1).sum()))
)
summary.to_string()

'                  n_significant  n_positive  n_negative\ncharacter_chosen                                       \nDEFECT                      250          67         183\nIRONCLAD                    290          54         236\nTHE_SILENT                  266          63         203\nWATCHER                     243          41         202'

In [13]:
top = (
    significant.groupby("character_chosen")
    .apply(lambda g: g.nlargest(5, "odds_ratio")[["card_name","n","odds_ratio","was_picked_pvalue"]], include_groups=False)
)
bottom = (
    significant.groupby("character_chosen")
    .apply(lambda g: g.nsmallest(5, "odds_ratio")[["card_name","n","odds_ratio","was_picked_pvalue"]], include_groups=False)
)
print("TOP 5 per character (best odds_ratio):")
print(top.to_string())
print()
print("BOTTOM 5 per character (worst odds_ratio):")
print(bottom.to_string())

TOP 5 per character (best odds_ratio):
                                  card_name         n  odds_ratio  was_picked_pvalue
character_chosen                                                                    
DEFECT           142            Concentrate     313.0    3.876976       3.382009e-03
                 95            Burning Pact     378.0    3.460880       8.585789e-04
                 443           Night Terror     343.0    2.357926       6.390193e-03
                 44               Barricade     344.0    1.956129       2.051981e-02
                 406   Master of Strategy+1    1206.0    1.947548       3.562503e-07
IRONCLAD         1096             Heatsinks     729.0    1.861871       4.248800e-02
                 1175  Master of Strategy+1    1749.0    1.860990       1.301075e-08
                 1234            Offering+1   31711.0    1.849281      1.663753e-134
                 1013             Expertise     689.0    1.835461       1.970774e-02
                 1174    M

In [ ]:
print(regression_pd.memory_usage(deep=True).sum() / 1e9, "GB in memory")
print(len(regression_pd), "rows")

In [15]:
regression_pd.to_parquet("../raw_data/win_rate_regression_input.parquet", index=False)
results_pd.to_parquet("../raw_data/card_win_rate_regression_results.parquet", index=False)